# 05 — Deploy to Amazon Bedrock AgentCore Runtime

**Production question:** *Can it run at scale, securely, without us managing servers?*

**AgentCore Runtime** hosts the agent:
- **Session isolation**: each customer session runs in its own microVM
- **Serverless**: scales with traffic, and you pay for what you use
- **Long-running**: sessions up to 8 hours
- **Built-in observability**: OpenTelemetry traces and logs in CloudWatch
- **IAM-based security**: the agent gets exactly the permissions you grant

```
 boto3 / app ──► AgentCore Runtime ──► main.py (BedrockAgentCoreApp)
                  (microVM / session)      └─► agent.py ─► tools · hooks · steering · skills
                                                         └─► AgentCore Memory
                                                         └─► Amazon Bedrock (model)
```

AgentCore deploys a **folder of code**, so this is where the notebook code becomes an app: `support_agent/`.

In [ ]:
import os, sys
for skill in ["refund-processing", "order-tracking", "account-troubleshooting"]:
    os.makedirs(f"support_agent/skills/{skill}", exist_ok=True)
sys.path.insert(0, "support_agent")

### Step 1 — Package the layers from notebooks 01–04 as modules

These are the **same cells** we ran earlier, now saved as files (collapsed, so expand them if you want to look). The only differences are imports between modules, and skills stored as `SKILL.md` files that the business team can edit.

In [ ]:
%%writefile support_agent/tools.py
"""ElectroBazaar backend tools (mock data standing in for real order / payment / ticket APIs)."""

from strands import tool

# --- Business constants (shared with the policy layer in steering.py) ---
RETURN_WINDOW_DAYS = 30
SUPERVISOR_REVIEW_THRESHOLD = 50_000  # ₹ — refunds at or above this need a human

# --- Mock data ---
CUSTOMERS = {
    "C-1001": {"name": "Priya Sharma", "city": "Bengaluru", "email": "priya.sharma@example.com",
               "phone": "+91 98450 12345", "account_status": "active"},
    "C-1002": {"name": "Arjun Reddy", "city": "Hyderabad", "email": "arjun.reddy@example.com",
               "phone": "+91 99490 67890", "account_status": "active"},
    "C-1003": {"name": "Kavya Iyer", "city": "Chennai", "email": "kavya.iyer@example.com",
               "phone": "+91 98410 24680", "account_status": "active"},
}

ORDERS = {
    "C-1001": [
        {"order_id": "ORD-5521", "item": "Wireless Earbuds", "amount": 3499, "status": "Delivered",
         "order_date": "2026-10-14", "delivered_date": "2026-10-18", "tracking": "AWB-7781229034"},
        {"order_id": "ORD-5488", "item": "USB-C Hub (7-in-1)", "amount": 2199, "status": "Shipped",
         "order_date": "2026-10-26", "estimated_delivery": "2026-10-31", "tracking": "AWB-7781229871"},
    ],
    "C-1002": [
        {"order_id": "ORD-5390", "item": "Mechanical Keyboard", "amount": 6999, "status": "Delayed",
         "order_date": "2026-10-15", "estimated_delivery": "2026-10-21", "tracking": "AWB-7781227745"},
    ],
    "C-1003": [
        {"order_id": "ORD-5301", "item": "Smartwatch", "amount": 12999, "status": "Delivered",
         "order_date": "2026-09-02", "delivered_date": "2026-09-06", "tracking": "AWB-7781221108"},
        {"order_id": "ORD-5602", "item": "Laptop (14-inch, 16 GB RAM)", "amount": 72990, "status": "Delivered",
         "order_date": "2026-10-20", "delivered_date": "2026-10-23", "tracking": "AWB-7781230456"},
    ],
}


def find_order(order_id: str) -> dict | None:
    for orders in ORDERS.values():
        for order in orders:
            if order["order_id"] == order_id:
                return order
    return None


def inr(amount: float) -> str:
    return f"₹{amount:,.0f}"


# --- Tools ---

@tool
def lookup_customer(customer_id: str) -> str:
    """Look up a customer's profile by their customer ID.

    Args:
        customer_id: The customer ID (e.g. C-1001)
    """
    c = CUSTOMERS.get(customer_id)
    if not c:
        return f"No customer found with ID {customer_id}"
    return (f"Customer: {c['name']} ({c['city']})\nEmail: {c['email']}\n"
            f"Phone: {c['phone']}\nAccount status: {c['account_status']}")


@tool
def get_order_history(customer_id: str) -> str:
    """Get the order history for a customer, including status, dates and tracking.

    Args:
        customer_id: The customer ID (e.g. C-1001)
    """
    orders = ORDERS.get(customer_id)
    if not orders:
        return f"No orders found for customer {customer_id}"
    lines = []
    for o in orders:
        line = f"{o['order_id']}: {o['item']} — {inr(o['amount'])} [{o['status']}] ordered {o['order_date']}"
        if o.get("delivered_date"):
            line += f", delivered {o['delivered_date']}"
        if o.get("estimated_delivery"):
            line += f", estimated delivery {o['estimated_delivery']}"
        lines.append(line + f", tracking {o['tracking']}")
    return "\n".join(lines)


@tool
def process_refund(order_id: str, amount: float) -> str:
    """Issue a refund for an order to the customer's original payment method.

    Args:
        order_id: The order ID to refund (e.g. ORD-5521)
        amount: The refund amount in Indian Rupees
    """
    return (f"Refund REF-{order_id[4:]} of {inr(amount)} initiated for {order_id} to the original "
            "payment method (UPI / card / net banking). It will reflect in 5-7 business days.")


@tool
def create_support_ticket(customer_id: str, reason: str, priority: str = "normal") -> str:
    """Escalate an issue to a human support specialist by creating a ticket.

    Args:
        customer_id: The customer ID (e.g. C-1001)
        reason: Short description of why this needs a human
        priority: "normal" or "high"
    """
    return (f"Ticket TKT-{abs(hash((customer_id, reason))) % 90000 + 10000} created for {customer_id} "
            f"({priority} priority). A specialist will respond within 24 hours.")


ALL_TOOLS = [lookup_customer, get_order_history, process_refund, create_support_ticket]

In [ ]:
%%writefile support_agent/prompts.py
"""System prompt and model configuration for the ElectroBazaar support agent."""

import os
from datetime import date

# Swapping models is a config change: e.g. amazon.nova-pro-v1:0 or a Claude Haiku profile
MODEL_ID = os.getenv("MODEL_ID", "global.anthropic.claude-sonnet-4-6")


def build_system_prompt() -> str:
    today = date.today().strftime("%d %B %Y (%A)")
    return f"""You are the customer support agent for ElectroBazaar, an online electronics store in India.
Today's date is {today}.

Be helpful, professional and concise. Use the available tools to look up customer
information and act on requests. All prices are in Indian Rupees (₹).

Guidelines:
- Always ask for the customer ID first if you don't have it.
- Verify the customer with lookup_customer before taking any action on their account.
- Use tool data to answer questions — never ask the customer for information you already have.
- If you have skills available, activate the one that matches the request and follow it.
- Remember preferences the customer shares (contact channel, refund method, language) and honour them.
- Be warm but efficient."""

In [ ]:
%%writefile support_agent/hooks.py
"""Deterministic guard rails for the support agent, implemented as Strands hooks."""

import json
import logging

from strands.hooks import (
    AfterToolCallEvent,
    BeforeInvocationEvent,
    BeforeToolCallEvent,
    HookProvider,
    HookRegistry,
)

logger = logging.getLogger("support_agent.audit")


class RateLimiterHook(HookProvider):
    """Caps each tool at max_calls per agent invocation."""

    def __init__(self, max_calls: int = 3):
        self.max_calls = max_calls
        self.counts: dict[str, int] = {}

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(BeforeInvocationEvent, self.reset)
        registry.add_callback(BeforeToolCallEvent, self.check)

    def reset(self, event: BeforeInvocationEvent) -> None:
        self.counts = {}

    def check(self, event: BeforeToolCallEvent) -> None:
        name = event.tool_use["name"]
        self.counts[name] = self.counts.get(name, 0) + 1
        if self.counts[name] > self.max_calls:
            # Cancelling returns this message to the model as the tool result
            event.cancel_tool = f"'{name}' hit the {self.max_calls}-call limit for this request. Do NOT call it again."
            print(f"[RATE LIMIT] 🚫 blocked {name} (call {self.counts[name]}/{self.max_calls})")


class AuditLogHook(HookProvider):
    """Records every tool call: name, input, status and latency."""

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AfterToolCallEvent, self.record)

    def record(self, event: AfterToolCallEvent) -> None:
        entry = {
            "tool": event.tool_use["name"],
            "input": event.tool_use["input"],
            "status": event.result["status"],
            "ms": round((event.duration or 0) * 1000),
        }
        logger.info(json.dumps(entry))
        print(f"[AUDIT] {json.dumps(entry, ensure_ascii=False)}")

In [ ]:
%%writefile support_agent/steering.py
"""ElectroBazaar policy enforcement for the support agent (Strands steering handlers)."""

from datetime import date

from pydantic import BaseModel, Field
from strands import Agent
from strands.vended_plugins.steering import (
    Guide,
    LedgerProvider,
    ModelSteeringAction,
    Proceed,
    SteeringHandler,
    ToolSteeringAction,
)

from tools import RETURN_WINDOW_DAYS, SUPERVISOR_REVIEW_THRESHOLD, find_order


class RefundPolicyHandler(SteeringHandler):
    """Deterministic refund policy, checked before every process_refund call."""

    name = "refund-policy"

    def __init__(self):
        super().__init__(context_providers=[LedgerProvider()])

    async def steer_before_tool(self, *, agent, tool_use, **kwargs) -> ToolSteeringAction:
        if tool_use.get("name") != "process_refund":
            return Proceed(reason="Not a refund")
        try:
            return self._check_refund(tool_use)
        except Exception as e:  # fail closed: a broken check must never approve a refund
            return self._guide(f"Refund policy check failed ({e}). Do not refund; create a support ticket instead.")

    def _check_refund(self, tool_use) -> ToolSteeringAction:
        ledger = self.steering_context.data.get("ledger") or {}
        tool_calls = ledger.get("tool_calls", [])

        def succeeded(tool_name: str) -> bool:
            return any(c["tool_name"] == tool_name and c["status"] == "success" for c in tool_calls)

        if not succeeded("lookup_customer"):
            return self._guide("Verify the customer with lookup_customer before processing a refund.")
        if not succeeded("get_order_history"):
            return self._guide("Check the order with get_order_history before processing a refund.")

        order_id = tool_use["input"].get("order_id")
        amount = float(tool_use["input"].get("amount", 0))
        order = find_order(order_id)
        if order is None:
            return self._guide(f"Order {order_id} does not exist. Confirm the order ID with the customer.")
        if amount > order["amount"]:
            return self._guide(f"Refund ₹{amount:,.0f} exceeds the order value ₹{order['amount']:,}.")
        if amount >= SUPERVISOR_REVIEW_THRESHOLD:
            return self._guide(
                f"Refunds of ₹{SUPERVISOR_REVIEW_THRESHOLD:,} or more need supervisor approval. "
                "Do not refund; create a high priority support ticket instead."
            )
        if order.get("delivered_date"):
            days = (date.today() - date.fromisoformat(order["delivered_date"])).days
            if days > RETURN_WINDOW_DAYS:
                return self._guide(
                    f"{order_id} was delivered {days} days ago, outside the {RETURN_WINDOW_DAYS}-day return window. "
                    "Do not refund; offer to raise a support ticket for a goodwill exception."
                )

        print(f"[POLICY] ✅ refund for {order_id} passed all checks")
        return Proceed(reason="Refund policy satisfied")

    @staticmethod
    def _guide(reason: str) -> Guide:
        print(f"[POLICY] ⛔ {reason}")
        return Guide(reason=reason)


class ToneReview(BaseModel):
    acceptable: bool = Field(description="True if the reply follows every guideline")
    feedback: str = Field(description="If not acceptable, specific guidance on what to fix")


class ToneGuardrailHandler(SteeringHandler):
    """LLM-based review of every final reply against ElectroBazaar's communication guidelines."""

    name = "tone-guardrail"
    GUIDELINES = """You review replies from an ElectroBazaar customer support agent (India).
A reply is acceptable only if it:
- Doesn't overpromise or guarantee timelines beyond what the system confirmed
- Doesn't blame the customer, the courier, or other departments
- Acknowledges the customer's frustration (if any) before jumping to solutions
- Is concise and actionable
- Never exposes internal system details, tool names or technical jargon"""

    def __init__(self, max_retries: int = 1):
        super().__init__()
        self.max_retries = max_retries
        self._retries = 0

    async def steer_after_model(self, *, agent, message, stop_reason, **kwargs) -> ModelSteeringAction:
        reply = "".join(block.get("text", "") for block in message["content"]).strip()
        if stop_reason != "end_turn" or not reply or self._retries >= self.max_retries:
            self._retries = 0
            return Proceed(reason="Nothing to review")

        reviewer = Agent(model=agent.model, system_prompt=self.GUIDELINES, callback_handler=None)
        result = await reviewer.invoke_async(f"Review this reply:\n\n{reply}", structured_output_model=ToneReview)
        review = result.structured_output
        if review.acceptable:
            print("\n[TONE] ✅ reply approved")
            self._retries = 0
            return Proceed(reason="Tone OK")

        self._retries += 1
        print(f"\n[TONE] ✏️  rewrite requested: {review.feedback}")
        return Guide(reason=f"Rewrite your last reply to the customer. Reviewer feedback: {review.feedback}")

In [ ]:
%%writefile support_agent/skills/refund-processing/SKILL.md
---
name: refund-processing
description: Step-by-step process for handling customer refund and return requests, including order verification, return window checks, supervisor review and refund issuance.
---

# Refund Processing

1. **Verify the customer** with `lookup_customer`.

2. **Find the order** with `get_order_history`. Never ask the customer for dates or amounts the system already has.

3. **Check the return window.** Orders can be refunded within 30 days of the `delivered` date. Compare it with today's date from your instructions.
   - If the window has closed, explain this kindly and offer to raise a `create_support_ticket` for a goodwill exception.

4. **Check the amount.** If the refund is ₹50,000 or more, do NOT call `process_refund`. Tell the customer a supervisor must approve it, create a `high` priority support ticket, and say they will hear back within 24 hours.

5. **Confirm with the customer.** State the order ID, item and refund amount in one line and ask for a yes/no.

6. **Process the refund** with `process_refund` only after the customer says yes.

7. **Set expectations.** Refunds go back to the original payment method (UPI / card / net banking) in 5-7 business days.

## Notes
- Be empathetic. The customer has a problem they want solved.
- Never refund more than the order value.

In [ ]:
%%writefile support_agent/skills/order-tracking/SKILL.md
---
name: order-tracking
description: Help customers check the status of their orders, understand delivery timelines, and handle delayed or missing shipments.
---

# Order Tracking

1. **Verify the customer** with `lookup_customer`.

2. **Get the order** with `get_order_history` and explain the status:
   - **Processing**: ships within 1-2 business days.
   - **Shipped**: share the AWB tracking number and estimated delivery date.
   - **Delivered**: confirm the delivery date.
   - **Delayed**: follow the delay steps below.

3. **If the order is delayed** (today is past the estimated delivery date):
   - Acknowledge the frustration and apologise sincerely.
   - Offer a choice: keep waiting with a ₹200 store credit, or cancel for a full refund.
   - If they want to wait, create a `normal` priority support ticket to chase the courier.

## Notes
- Always share the tracking number when available.
- If an order shows "Delivered" but the customer hasn't received it, create a `high` priority ticket and promise a response within 24 hours.

In [ ]:
%%writefile support_agent/skills/account-troubleshooting/SKILL.md
---
name: account-troubleshooting
description: Guide for helping customers with account issues including login problems, OTP issues, locked accounts and updating contact details.
---

# Account Troubleshooting

1. **Verify the customer** with `lookup_customer`. Confirm their name and registered email match before discussing account changes.

2. **Identify the issue:**
   - **Not receiving OTP**: check that the registered mobile number is correct and ask them to wait 60 seconds before retrying. If it still fails, create a support ticket.
   - **Locked account** (5 failed attempts): explain that it unlocks automatically after 30 minutes, or create a ticket for an immediate unlock.
   - **Update phone/email**: changes need verification on both the old and new contact. Create a support ticket to start the change.

## Notes
- Never reveal full contact details or payment information. Mask them (e.g. +91 98xxx xx345).
- If the customer can't verify basic details, politely escalate to the security team for their protection.

Point the app at the AgentCore Memory we created in notebook 04 (`create_or_get_memory` returns the existing one):

In [ ]:
import logging
import boto3
from bedrock_agentcore.memory import MemoryClient

logging.getLogger("bedrock_agentcore.memory").setLevel(logging.CRITICAL)  # quiet the "already exists" log on reuse
REGION = boto3.session.Session().region_name or "us-east-1"
MEMORY_ID = MemoryClient(region_name=REGION).create_or_get_memory(name="ElectroBazaarSupportMemory")["id"]

with open("support_agent/memory_config.py", "w") as f:
    f.write("# AgentCore Memory settings (generated by 05_deploy_to_agentcore.ipynb)\n\n")
    f.write(f'MEMORY_ID = "{MEMORY_ID}"\nMEMORY_REGION = "{REGION}"\n')
print(open("support_agent/memory_config.py").read())

### Step 2 — `agent.py`: one factory that assembles every layer

In [ ]:
%%writefile support_agent/agent.py
"""Assembles the production ElectroBazaar support agent from all of its layers."""

import logging
from pathlib import Path

from strands import Agent, AgentSkills
from strands.models import BedrockModel

from hooks import AuditLogHook, RateLimiterHook
from prompts import MODEL_ID, build_system_prompt
from steering import RefundPolicyHandler, ToneGuardrailHandler
from tools import ALL_TOOLS

logger = logging.getLogger(__name__)
SKILLS_DIR = Path(__file__).parent / "skills"


def _memory_session_manager(session_id: str, actor_id: str):
    try:
        from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
        from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager
        from memory_config import MEMORY_ID, MEMORY_REGION
    except ImportError:
        logger.warning("AgentCore Memory not configured; running without persistent memory")
        return None

    config = AgentCoreMemoryConfig(
        memory_id=MEMORY_ID,
        actor_id=actor_id,
        session_id=session_id,
        retrieval_config={
            "/preferences/{actorId}/": RetrievalConfig(top_k=5, relevance_score=0.3),
            "/facts/{actorId}/": RetrievalConfig(top_k=5, relevance_score=0.3),
        },
    )
    return AgentCoreMemorySessionManager(config, region_name=MEMORY_REGION)


def build_agent(session_id: str | None = None, actor_id: str | None = None) -> Agent:
    session_manager = _memory_session_manager(session_id, actor_id) if session_id and actor_id else None
    return Agent(
        name="electrobazaar-support",
        model=BedrockModel(model_id=MODEL_ID),
        tools=ALL_TOOLS,
        hooks=[RateLimiterHook(), AuditLogHook()],
        plugins=[AgentSkills(skills=[str(SKILLS_DIR)]), RefundPolicyHandler(), ToneGuardrailHandler()],
        system_prompt=build_system_prompt(),
        context_manager="auto",  # offloads large tool results, summarises old turns near the context limit
        session_manager=session_manager,
    )

### Step 3 — `main.py`: the AgentCore entrypoint

`BedrockAgentCoreApp` turns any Python function into an AgentCore-compatible HTTP service (`/invocations` + `/ping`). AgentCore passes the **runtime session ID** in `context`, and we use it as the memory session.

In [ ]:
%%writefile support_agent/main.py
"""AgentCore Runtime entrypoint for the ElectroBazaar support agent."""

import json
import logging

from bedrock_agentcore.runtime import BedrockAgentCoreApp

from agent import build_agent

logging.basicConfig(level=logging.INFO)
app = BedrockAgentCoreApp()
_agents = {}  # one agent per runtime session


@app.entrypoint
def invoke(payload, context):
    prompt = payload.get("prompt")
    try:  # `agentcore invoke` may send the whole payload as a JSON string
        parsed = json.loads(prompt)
        payload, prompt = parsed, parsed.get("prompt", prompt)
    except (TypeError, json.JSONDecodeError):
        pass
    if not prompt:
        raise ValueError("Missing required field: prompt")

    session_id = context.session_id or "local-session"
    actor_id = payload.get("customer_id", "anonymous")
    if session_id not in _agents:
        _agents[session_id] = build_agent(session_id=session_id, actor_id=actor_id)

    result = _agents[session_id](prompt)
    return str(result).strip()


if __name__ == "__main__":
    app.run()

In [ ]:
%%writefile support_agent/requirements.txt
strands-agents>=1.42.0
bedrock-agentcore[strands-agents]>=1.13.0
aws-opentelemetry-distro>=0.10.1
boto3

### Step 4 — Smoke test locally (same code that will run in the cloud)

In [ ]:
from agent import build_agent

agent = build_agent()   # no memory for the smoke test
result = agent("Hi, I'm C-1001. When will my USB-C hub arrive?")
print(f"\n📊 Tokens: {result.metrics.accumulated_usage['totalTokens']}")

In [ ]:
# Our complete production app
for root, _, files in sorted(os.walk("support_agent")):
    if "__pycache__" in root:
        continue
    for f in sorted(files):
        print(os.path.join(root, f))

### Step 5 — Deploy with the AgentCore CLI (run in a terminal)

From this folder:

```bash
# 1. Create the AgentCore project (scaffolds the project + its CDK infrastructure code)
agentcore create --name electrobazaar
cd electrobazaar

# 2. Add a code-based Runtime for our agent
agentcore add runtime --name SupportAgent --language python

# 3. Copy in the app we just packaged
cp -r ../support_agent/* app/SupportAgent/

# 4. Dependencies (agentcore builds from pyproject.toml)
cd app/SupportAgent
uv init --bare --python 3.13
uv add -r requirements.txt
cd ../..

# 5. Test locally, then ship
agentcore dev        # local server that mimics the runtime (Ctrl+C to stop)

curl -s -X POST localhost:8080/invocations -H 'Content-Type: application/json' \
  -H 'X-Amzn-Bedrock-AgentCore-Runtime-Session-Id: priya-local-test-session-0000000001' \
  -d '{"prompt": "Hi, I am C-1001. Where is my USB-C hub?", "customer_id": "C-1001"}'

  
agentcore deploy     # zips to S3, provisions the runtime via CloudFormation (~2-3 min)
agentcore status     # copy the runtime ARN

# 6. Quick test of the deployed agent from the CLI
agentcore invoke --runtime SupportAgent --payload '{"prompt": "Hi, I am C-1001. Where is my USB-C hub?", "customer_id": "C-1001"}'
```

Behind `agentcore deploy`: the CLI installs ARM64 Linux wheels with `uv`, zips them with our code to S3 (no Docker needed), an **IAM execution role** is created, and **CloudWatch** logging and tracing are wired up.

### Step 6 — Least privilege: let the runtime use *our* memory

The execution role created by the CLI can call Bedrock models. We explicitly grant it access to **this one memory resource** and nothing else.

In [ ]:
import json

AGENT_ARN = "arn:aws:bedrock-agentcore:us-east-1:829250931565:runtime/electrobazaar_default_SupportAgent-Oe6QexGTX3"  # ← paste from `agentcore status`

account = boto3.client("sts").get_caller_identity()["Account"]
control = boto3.client("bedrock-agentcore-control", region_name=REGION)
runtime = control.get_agent_runtime(agentRuntimeId=AGENT_ARN.split("/")[-1])
role_name = runtime["roleArn"].split("/")[-1]

boto3.client("iam").put_role_policy(
    RoleName=role_name,
    PolicyName="ElectroBazaarMemoryAccess",
    PolicyDocument=json.dumps({
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Action": [
                # exactly the APIs the AgentCore Memory session manager calls
                "bedrock-agentcore:CreateEvent", "bedrock-agentcore:GetEvent", "bedrock-agentcore:ListEvents",
                "bedrock-agentcore:DeleteEvent", "bedrock-agentcore:RetrieveMemoryRecords",
            ],
            "Resource": f"arn:aws:bedrock-agentcore:{REGION}:{account}:memory/{MEMORY_ID}",
        }],
    }),
)
print(f"✅ Granted memory access to {role_name}")

### Step 7 — Invoke the deployed agent (the production path)

Your website, mobile app or WhatsApp bot calls the runtime with the AWS SDK. Same `runtimeSessionId` = same conversation (and the same microVM).

In [ ]:
import uuid

client = boto3.client("bedrock-agentcore", region_name=REGION)


def ask(prompt: str, session_id: str, customer_id: str) -> str:
    response = client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_ARN,
        runtimeSessionId=session_id,  # must be 33+ characters
        payload=json.dumps({"prompt": prompt, "customer_id": customer_id}).encode(),
        qualifier="DEFAULT",
    )
    text = "".join(chunk.decode("utf-8") for chunk in response["response"])
    try:
        text = json.loads(text)  # the runtime returns the reply as a JSON string
    except json.JSONDecodeError:
        pass
    print(f"🤖 {text}\n")


session = f"arjun-{uuid.uuid4()}"
ask("Hi, I'm C-1002. My keyboard still hasn't arrived.", session, "C-1002")
ask("I'll wait — please add the credit.", session, "C-1002")

In [ ]:
# Kavya, in a brand new session. Long-term memory from notebook 04 follows her into production.
ask("Hi, it's Kavya (C-1003). How will you contact me if there's an update on my laptop?",
    f"kavya-{uuid.uuid4()}", "C-1003")

### Step 8 — Observe it

Open **CloudWatch → GenAI Observability → Bedrock AgentCore**. For every request you'll see:
- the full trace: model calls, tool calls, latency per step, token usage
- our `[AUDIT]` and `[POLICY]` log lines from `hooks.py` and `steering.py`

Or straight from the terminal:

```bash
agentcore logs runtime --name SupportAgent --since 15m --query POLICY   # our policy decisions
agentcore traces runtime list --name SupportAgent                      # recent request traces
```

(One-time setup: enable **CloudWatch Transaction Search** in the account. `aws-opentelemetry-distro` in `requirements.txt` handles the instrumentation.)

### Cleanup

```bash
agentcore remove all --yes   # resets local config
agentcore deploy          # applies the teardown in AWS
```
```python
MemoryClient(region_name=REGION).delete_memory(MEMORY_ID)   # optional: delete AgentCore Memory
```

---

## 🎯 What we built, end to end

| Layer | Production concern | Built with |
|---|---|---|
| Tools | Acts on real business data | Strands `@tool` |
| Hooks | Cost control and audit trail | Strands hooks |
| Skills | Business-owned SOPs | Strands `AgentSkills` |
| Steering | Refund policy enforced in code, tone checked by an LLM | Strands steering |
| Memory | Survives restarts, remembers customers | **AgentCore Memory** |
| Runtime | Serverless, session-isolated, IAM-secured | **AgentCore Runtime** |
| Observability | Traces, logs, metrics | **AgentCore + CloudWatch** |

### Where to go next
- **AgentCore Gateway**: expose your existing APIs and Lambdas as MCP tools
- **AgentCore Identity**: let the agent act on behalf of the signed-in customer (OAuth)
- **Multi-agent**: delegate device issues to a tech-support specialist agent
- **Evals**: test every release automatically with LLM-as-judge
- **Bedrock Guardrails**: PII redaction and content filters on every model call